# Predicting Sustained Snow Disappearance from a 1 March Forecast Origin

CODS-622 Machine Learning, course project.

**Run order:** top to bottom. Every cell reads from files committed to the repository, so
nothing here depends on the NRCS service being reachable.

**Sections map to the rubrics:** setup, data and audit, target, features and leakage,
splits, baselines, models, evaluation, ablations, demo.

## 0. Setup

Clones the repository and pins dependencies. Skip the clone if running locally.

In [ ]:
# Colab only. Comment out when running locally.
!git clone -q https://github.com/AlShamsiK/snow-disappearance.git
%cd snow-disappearance
!pip install -q -r requirements.txt

In [ ]:
import sys, platform, json
import numpy as np
import pandas as pd
import sklearn, lightgbm

sys.path.insert(0, ".")
from src import config, clean, target, features, models, evaluate

np.random.seed(config.RANDOM_SEED)

print("python:", platform.python_version())
print("numpy:", np.__version__, "| pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__, "| lightgbm:", lightgbm.__version__)
print()
print("configuration:")
for k, v in config.as_dict().items():
    print(f"  {k} = {v}")

## 1. Data

Source: USDA NRCS National Water and Climate Center, SNOTEL / SCAN network.

Raw files in `data/raw/` were downloaded once on `TBD` and are committed unmodified.
`src/data_download.py` documents how they were obtained but is not run here.

In [ ]:
# Load the committed raw daily records and station metadata.
raw = None       # pd.read_csv(config.DATA_RAW / "...")
stations = None  # pd.read_csv(config.DATA_RAW / "stations.csv")


### 1.1 Data audit

Before any cleaning: how much is missing, are there duplicate station-date rows, are the
identifiers consistent, and are there values that cannot be physically right.

In [ ]:
# audit = clean.audit_raw(raw)
# audit

### 1.2 Cleaning and screening

Each decision below is recorded with the number of rows or station-years it removes, so
the write-up can justify it rather than assert it.

Known SNOTEL failure modes to handle: negative SWE from pillow drift, sudden steps from
sensor replacement, flat-lined stretches, isolated zeros inside a continuous snowpack, and
long gaps that make a station-year unusable.

In [ ]:
# daily = clean.drop_duplicates(raw)
# daily = clean.flag_swe_anomalies(daily)
# daily = clean.fill_missing(daily)
# daily, exclusions = clean.screen_stations(daily)
# exclusions

## 2. Target: sustained snow disappearance

The first day on or after 1 March when SWE falls to or below the threshold and stays there
for the required number of consecutive days.

The "sustained" requirement is what distinguishes a genuine melt-out from a brief bare-ground
spell, a sensor dropout, or a late-season snowfall that vanishes within days. Both parameters
are configurable and their influence is tested in the ablation section.

Station-years where the pack never disappears, or where the record is too incomplete to tell,
are recorded as such rather than dropped silently.

In [ ]:
# targets = target.build_targets(daily)
# targets.head()

In [ ]:
# Distribution of the target, and how many station-years are censored or unusable.


## 3. Features as of 1 March

**Leakage rule:** a feature may use data dated strictly before 1 March of its own water year,
static station attributes, or statistics estimated on training water years only. Nothing else.

Three traps specific to this problem:
1. Using any observation from March onward, which is the whole forecasting task.
2. Station climatology computed over the full record, including the test years.
3. Percent-of-median fields taken from the data source, whose medians are computed over a
   period of record that includes the test years.

In [ ]:
# X = features.build_feature_table()
# features.assert_no_leakage(X, daily)
# X.head()

## 4. Splits

Random k-fold would be wrong here. Station-years are correlated through time and across
nearby stations, so a random split leaks both the future and the neighbours.

Three evaluation regimes:
- **Forward chaining over water years.** Train on the past, predict forward. This is the
  operational question.
- **Leave-one-station-out.** Can the model handle a station it has never seen.
- **Spatial blocks.** Hold out whole regions, to check that apparent skill is not just
  spatial autocorrelation.

The most recent block of water years is held out entirely and touched once, at the end.

In [ ]:
# splits = evaluate.forward_chaining_splits(X)

## 5. Baselines

The model has to beat these or the project has no finding.

- **Climatology.** Each station's historical average disappearance date, from training years only.
- **Degree-day melt model.** Physically motivated: melt the 1 March snowpack at a calibrated rate.
- **SWE-only linear regression.** How much of the answer is simply how much snow is on the ground.

In [ ]:
# clim = models.ClimatologyBaseline().fit(X_train, y_train)
# dd = models.DegreeDayBaseline().fit(X_train, y_train)
# swe_lin = models.SWEOnlyLinearBaseline().fit(X_train, y_train)

## 6. Models

Ridge, random forest, and gradient boosting, with hyperparameters tuned inside the training
folds only. Quantile regressors supply the prediction interval used in the demo.

## 7. Evaluation

Headline metrics in days: MAE, RMSE, and signed bias. The number that matters most is the
skill score against climatology, because an MAE that sounds good in isolation may be no
better than the long-run average for that station.

Bias is reported separately from MAE: a model that is unbiased on average but systematically
late in warm springs is failing differently from one that is simply noisy.

## 8. Ablations and robustness

- Feature groups: snowpack only, plus accumulation-season weather, plus static attributes.
- Sensitivity to the target definition: vary the sustain window and the SWE threshold.
- Error by elevation band.
- Performance in low-snow versus high-snow years.
- Generalisation to unseen stations and unseen years.

### Failure cases to look for
Low-elevation stations with intermittent snowpack, late-season storms, rain-on-snow events,
unusually warm springs, and stations near the rain-snow transition.

## 9. Demo

Pick a station and a water year, show the features exactly as they stood on 1 March, predict
the disappearance date with an interval, and compare it against what actually happened.

The model is loaded from `models/`, not retrained.

In [ ]:
import joblib

# model = joblib.load(config.MODELS / "model.joblib")

STATION = None      # e.g. "301:CA:SNTL"
WATER_YEAR = None   # e.g. 2019

# 1. Show the inputs available on 1 March of that water year
# 2. Predict the disappearance date and interval
# 3. Show the observed date and the error in days

## 10. Limitations

To be written up alongside the report. Candidates: the target definition is a choice and
results move with it, the station set is a subset rather than the full network, prediction
skill will degrade at low elevations, and the model is fitted to historical climate and
carries no guarantee under conditions outside the training record.